<a href="https://colab.research.google.com/github/supurazako/ml-security-jp/blob/master/ch02/Chapter2_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![表紙](https://www.oreilly.co.jp/books/images/picture978-4-87311-907-6.gif)

このノートブックはオライリー・ジャパンより発行の書籍[『セキュリティエンジニアのための機械学習』](https://www.oreilly.co.jp/books/9784873119076/)のサンプルコードです。コードの解説等は書籍をご参照ください。なお、このコードを動作させた結果について、著者およびオライリー・ジャパンは一切の責任を負いません。

## データセットの用意

### データセットのダウンロード

ここでポイント. 元のURL: `http://www.aueb.gr/users/ion/data/lingspam_public.tar.gz` は`301 Moved Permanently`で `http://pages.aueb.gr/users/ion/data/lingspam_public.tar.gz` に飛ばされ, そこも301が出て, `https://www2.aueb.gr/users/ion/data/lingspam_public.tar.gz` に飛ばされます.

ここで以下のようなエラーが出ます.
```
ERROR: cannot verify www2.aueb.gr's certificate, issued by ‘CN=GEANT OV RSA CA 4,O=GEANT Vereniging,C=NL’:
  Unable to locally verify the issuer's authority.
To connect to www2.aueb.gr insecurely, use `--no-check-certificate'.
```

`ca-certificates` をupdateしてもだめだったので、今回は `--no-check-certificate` を使います。

In [12]:
!wget --no-check-certificate http://www.aueb.gr/users/ion/data/lingspam_public.tar.gz

--2025-08-22 12:02:41--  http://www.aueb.gr/users/ion/data/lingspam_public.tar.gz
Resolving www.aueb.gr (www.aueb.gr)... 195.251.255.156
Connecting to www.aueb.gr (www.aueb.gr)|195.251.255.156|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: http://pages.aueb.gr/users/ion/data/lingspam_public.tar.gz [following]
--2025-08-22 12:02:41--  http://pages.aueb.gr/users/ion/data/lingspam_public.tar.gz
Resolving pages.aueb.gr (pages.aueb.gr)... 195.251.255.230
Connecting to pages.aueb.gr (pages.aueb.gr)|195.251.255.230|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www2.aueb.gr/users/ion/data/lingspam_public.tar.gz [following]
--2025-08-22 12:02:41--  https://www2.aueb.gr/users/ion/data/lingspam_public.tar.gz
Resolving www2.aueb.gr (www2.aueb.gr)... 195.251.255.230
Connecting to www2.aueb.gr (www2.aueb.gr)|195.251.255.230|:443... connected.
  Unable to locally verify the issuer's authority.
HTTP request 

### データセットの解凍

In [13]:
!tar -zxf ./lingspam_public.tar.gz

### データセットの確認

In [14]:
!ls ./lingspam_public/bare/part3/

6-1056msg2.txt	6-1109msg1.txt	6-1151msg1.txt	6-16msg1.txt   6-228msg1.txt
6-1056msg3.txt	6-1109msg2.txt	6-1152msg1.txt	6-173msg1.txt  6-22msg1.txt
6-1058msg1.txt	6-1109msg3.txt	6-1154msg1.txt	6-173msg2.txt  6-232msg1.txt
6-1059msg1.txt	6-110msg1.txt	6-1155msg1.txt	6-173msg3.txt  6-232msg2.txt
6-1061msg1.txt	6-110msg2.txt	6-1155msg2.txt	6-175msg1.txt  6-232msg3.txt
6-1062msg1.txt	6-110msg3.txt	6-1155msg3.txt	6-176msg1.txt  6-235msg1.txt
6-1063msg1.txt	6-1111msg1.txt	6-1156msg1.txt	6-182msg1.txt  6-236msg1.txt
6-1063msg2.txt	6-1112msg1.txt	6-1157msg1.txt	6-186msg1.txt  6-237msg1.txt
6-1063msg3.txt	6-1112msg2.txt	6-1157msg2.txt	6-186msg2.txt  6-241msg1.txt
6-1065msg1.txt	6-1112msg3.txt	6-1157msg3.txt	6-186msg3.txt  spmsga36.txt
6-1066msg1.txt	6-1114msg1.txt	6-1159msg1.txt	6-187msg1.txt  spmsga37.txt
6-1069msg1.txt	6-1115msg1.txt	6-1161msg1.txt	6-188msg1.txt  spmsga38.txt
6-1070msg1.txt	6-1116msg1.txt	6-118msg1.txt	6-189msg1.txt  spmsga39.txt
6-1071msg1.txt	6-1117msg1.txt	6-119msg1.txt	6-1

## ラベルの作成

迷惑メールとそうでないものに分別し、ラベルを追加する。

pandas:表形式でデータを扱うのに長けたライブラリ。

In [23]:
import os
import glob
import pandas as pd

path = "./lingspam_public/bare/"

text = []
label = []

# part?ディレクトリ配下にあるメールデータを読み込み、
# 迷惑メールとそうでないものを分別してラベルを追加
for part in range(1,10):
    folder = os.path.join(path, 'part'+str(part))
    for filePath in glob.glob(os.path.join(folder, '*.txt')):
        if ('spmsga' in filePath):
            with open(filePath) as f:
                text.append(f.read())
                label.append(1)
        else:
            with open(filePath) as f:
                text.append(f.read())
                label.append(0)

pandasのDataFrameの作成

DataFrameはvalues, columns, indexの3つからなる2次元の表形式(Excelやスプレッドシートのようなものをイメージすると分かりやすい)。

```python
data = pd.DataFrame()
```
ここでは、空のデータフレームを作っている。

```python
data['Text'] = text
data['label'] = label
```
ここでは、`Text`というrowを作成し、そこに`text`を入れている。`label`も同様。

c.f. https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html

In [19]:
data = pd.DataFrame()
data['Text'] = text
data['label'] = label

### tf-ifd

#### `TfidfVectorizer`

tf-idf: 「レアな単語が何回も出てくるようなら、文書を分類する際にその単語の重要度を上げる」という手法。

tfは「term frequency」、つまり単語の出現頻度です。各文書でどのくらいの頻度で単語が出たかということ。

idfは「inverse document frequency」、逆文書頻度と呼ばれるもの。単語がレアなら高い値を、色々な文書でよく出てくる単語ならば低い値を示すというもの。

この2つを組み合わせたのが「tf-idf」。つまり「その単語がよく出現するほど」、「その単語がレアなほど」大きい値を示すものだ。

tf-idfを使うと文章の特徴を判別することができる。また、文書間の類似度もtf-idfで判別できる。

(引用: https://dev.classmethod.jp/articles/yoshim_2017ad_tfidf_1-2/)

そんなtf-idfの計算を勝手にやってくれるのが`TfidfVectorizer`だ。

#### stop_words

`a`や`the`, `is` といった文章の意味にあまり影響しない一般的な単語を除外する設定。これをすることにより、ノイズが減り、より重要な単語にフォーカスできる。今回は`english`を選択しているので、英語における一般的な単語を除外する。

#### fit_transform

`fit`と`transform`を同時に行っている。

`fit`では、どのような単語が存在するのかを学習し、「単語の辞書」を作成する。また、各単語のidf値もここで計算する。

`transform`では、学習した辞書とidf値を使って、各メール本文をtf-idfスコアのベクトルに変換する。結果として得られるXは、スパース行列という形式である。

##### スパース行列とは

sparse(まばらな)行列。日本語では疎行列と呼ぶ。

成分のほとんどが0である行列のこと。列のゼロ要素の数を要素数の合計で割った値を、行列のスパース性と呼ぶことがある。

tf-idfでは、0がよく出てくる。

分かりやすい例を引用

> 例えば、`あめんぼ赤いなあいうえお`と`あめんぼは赤くないのでわ？	`という文章があるとします。この文書群に対して以下のようにTF-IDFを計算すると、列に全ての文書中にある単語、各文書が1つの行。文書に含まれない単語は0、文書に含まれる単語は 0<x<=1 の値となるテーブルが作られます。（ここでは１文書が１文となってますが、実際は文書中の文の数や長さは問いません）

|文書|あいうえお|あめんぼ|ない|ので|赤い|赤く|
|-|-|-|-|-|-|-|
|あめんぼ赤いなあいうえお|0.632|0.449|0.000|0.000|0.632|0.000|
|あめんぼは赤くないのでわ？|0.000|0.380|0.534|0.534|0.000|0.534|

> 値が各文書における単語の重要度なのですが、それは置いておいて、計算対象となる文書とそれに含まれる単語が増えていったときに疎行列になりそうだな〜ということは分かってもらえたでしょうか。

引用: https://techtekt.persol-career.co.jp/entry/tech/20231205_01

上記のように、tf-idfは疎行列で扱うのに向いており、疎行列を使うことでメモリの節約になる。

#### その後の処理

```python
X = pd.DataFrame(X.toarray())
X = X.astype('float')
```
スパース行列を通常の行列形式に変換し、DataFrameを作成する。

その後、DataFrameの値をすべてfloatに変換。

```python
X.columns = column_names
y = data['label']
```

列名を単語のリストに置き換える

迷惑メールか否かのラベルをつける

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TfidfVectorizerを初期化する。
# stop_wordsにenglishを指定し、一般的な単語を除外する
tfidf = TfidfVectorizer(stop_words="english")

X = tfidf.fit_transform(data['Text'])
column_names = tfidf.get_feature_names_out()

# Xにベクトル化した値を整形して代入
X = pd.DataFrame(X.toarray())
X = X.astype('float')
# カラム名を設定
X.columns = column_names
y = data['label']

以下がそのDataFrame

In [21]:
X

,00,000,0000,00001,00003000140,00003003958,0001,00010,00014,0003,...,zwischen,zwitserlood,zxgah7qabjh,zybatov,zybatow,zygmunt,zyokyoozyu,zytkow,zz214,zzlsa
0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2597,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2598,0.168895,0.195922,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2599,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2600,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


optuna(ハイパーパラメータチューニングを自動でするライブラリ)をinstall

In [26]:
!pip install optuna-integration[lightgbm]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 2.5 MB/s eta 0:00:00


## LightGBMを使用する検出器

### 流れ

1. データセットを訓練用とテスト用に分割
2. パラメータを設定
3. 交差検証を使用し、ハイパーパラメータを探索

### 交差検証とは

データを分割し、一部分を学習に使い、残りでテストを行う。←これを分割数の数だけやる

ホールドアウト検証と違う点は、3分割以上するという点にある。これにより、どの部分を学習すると一番精度が高いかを検証することが可能になる

なお、1.で行われている分割は交差検証のためではない。

以下のリンクでは、今回使用している`LightGBMTunerCV`の使用法が解説されている。これの`nfold=5`という部分が何回検証を行うのかということだ。初期値では5回に設定されている。

https://optuna-integration.readthedocs.io/en/v4.5.0_a/reference/generated/optuna_integration.lightgbm.LightGBMTunerCV.html

### LightGBMとは

LightGBMは2016年にmicrosoftによって公開された勾配ブースティング木である。

### 勾配ブースティング木

勾配ブースティングとは、ブースティングアルゴリズムのひとつのこと。ブースティングとは、集団学習のフレームワークのひとつであり、複数の弱学習器(単独で使うには精度の低い学習器のこと)を統合して全体の学習器を構成する手法(アンサンブル学習)をとる。勾配ブースティング木では、この弱学習器に決定木を活用している。

### パラメータ

Optunaが調整するパラメータではなく、今回のタスクの前提条件を指定する。

`"objective": "binary"`:今回の目的は「二値分類b」つまり迷惑メールか否かであることを指定します。

`"metric": "binary_logloss"`: モデルの性能を評価するための指標（Metric）。値が小さいほど、モデルの予測が正確であることを意味する。Optunaはこの値を最小化しようとする。

`"verbosity": -1`: 学習中の詳細なログを抑制する。しかし、このパラメータは現在の安定版では削除されているため、代替の`set_verbosity`を使用することが推奨される。([公式doc v3.5.1](https://optuna.readthedocs.io/en/v3.5.1/reference/generated/optuna.integration.lightgbm.LightGBMTunerCV.html))

`"boosting_type": "gbdt"`: LightGBMのアルゴリズムの種類を指定だと推測される。

### 交差検証の実行

学習の反復回数の上限を100回にし、学習を行わせる

In [27]:
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
import optuna.integration.lightgbm as olgb
import optuna

# データセットを訓練用とテスト用に分割
X_train, X_test, y_train, y_test = \
train_test_split(X, y, test_size=0.2, shuffle=True, random_state=101)

# LightGBM用のデータセットに変換
train = olgb.Dataset(X_train, y_train)

# パラメータの設定
params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "verbosity": -1,
    "boosting_type": "gbdt",
}

# 交差検証を使用したハイパーパラメータの探索
tuner = olgb.LightGBMTunerCV(params, train, num_boost_round=100)

# ハイパーパラメータ探索の実行
tuner.run()

[I 2025-08-22 12:12:43,373] A new study created in memory with name: no-name-2e1e747a-4c00-443f-ac91-3992d32cb840
min_child_samples, val_score: 0.113614: 100%|##########| 5/5 [08:31<00:00, 102.31s/it]


## 訓練

この節では、ハイパーパラメータチューニングによって見つけられた最適な値を用い、実際にLightGBMモデルを訓練し、それを評価する。

### 混同行列

|||予測結果||
|-|-|-|-|
|||陰性|陽性|
|正解データ|陰性|真陰性(TN)|偽陽性(FP)|
||陽性|偽陰性(FN)|真陽性(TP)|

端的に言えば上の表のことだ。少し解説すると

- 真陽性(TP): フィッシングサイトをフィッシングサイトであると正しく検出できている数
- 偽陰性(FN): フィッシングサイトを見逃してしまった数
- 真陰性(TN): 非フィッシングサイトを正しく検出できた数
- 偽陽性(FP): 非フィッシングサイトをフィッシングサイトと誤検知した数

という意味だ。

### 結果

```
Accuracy: 93.85797 %
[[476   2]
 [ 30  13]]
 ```

In [29]:
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

# 訓練データとテストデータを設定
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test)

# ハイパーパラメータ探索で特定した値を設定
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'lambda_l1': tuner.best_params['lambda_l1'],
    'lambda_l2': tuner.best_params['lambda_l2'],
    'num_leaves': tuner.best_params['num_leaves'],
    'feature_fraction': tuner.best_params['feature_fraction'],
    'bagging_fraction': tuner.best_params['bagging_fraction'],
    'bagging_freq': tuner.best_params['bagging_freq'],
    'min_child_samples': tuner.best_params['min_child_samples']
}

# 訓練の実施
gbm = lgb.train(
    params,
    train_data,
    num_boost_round=100,
)

# テスト用データを使って予測する
preds = gbm.predict(X_test)
# 返り値は確率になっているので四捨五入する
pred_labels = np.rint(preds)
# 正解率と混同行列の出力
print("Accuracy: {:.5f} %".format(100 * accuracy_score(y_test, pred_labels)))
print(confusion_matrix(y_test, pred_labels))

Accuracy: 93.85797 %
[[476   2]
 [ 30  13]]
